# 03 — Modelado Predictivo (v2)

**Objetivo:** Entrenar y comparar múltiples modelos de clasificación binaria para predecir si un lead será Hot (1) o Cold (0), utilizando el dataset ya transformado por el pipeline V2.

**Modelos evaluados:**
1. Logistic Regression (baseline)
2. Random Forest
3. Gradient Boosting
4. LightGBM

**Pasos de este notebook:**
1. Cargar datasets procesados
2. Entrenar un baseline lineal
3. Entrenar modelos basados en árboles
4. Comparar métricas globales
5. Analizar curvas ROC y matrices de confusión
6. Revisar importancia de variables
7. Validar estabilidad con cross-validation
8. Guardar el mejor modelo

---

### Cambios respecto a v1

| Aspecto | v1 | v2 |
|---|---|---|
| **Features** | 48 (one-hot encoding) | **11** (features densas y Target Encoding) |
| **Problema corregido** | Modelos propensos a fijarse en ruidos estadísticos de categorías pequeñas (ej: 100% conversión con n=2) | El modelo usa Target Encoding con suavizado bayesiano para protegerse de muestras pequeñas |
| **Encoding** | 1 columna binaria por categoría | Reemplazo in-place por tasas suavizadas; menor dimensionalidad y árboles más rápidos |
| **Interpretabilidad** | Importancias fragmentadas entre muchas columnas binarias | Señales más compactas y comparables entre variables |
| **Robustez** | Mayor riesgo de sesgo por volumen | Menor sobreajuste y mejor generalización esperada |

## 1. Cargar datos preprocesados (v2)

En este paso cargamos los datasets exportados por `02_feature_engineering_v2.ipynb`. El objetivo es partir de una representación ya consistente entre train y test, con las transformaciones supervisadas aplicadas correctamente solo sobre entrenamiento.

**Qué validar aquí:**
- que `X_train` y `X_test` tengan exactamente las mismas columnas
- que la proporción de Hot Leads sea similar entre train y test
- que la reducción de dimensionalidad lograda en V2 se conserve en los archivos exportados

**Diferencias V1 vs V2:**
En la V1 el modelado trabajaba con una matriz mucho más dispersa por el uso intensivo de One-Hot Encoding. En la V2 arrancamos desde un dataset más compacto, lo que reduce costo computacional y facilita comparaciones entre modelos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
)
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

X_train = pd.read_csv("../data/processed/X_train_v2.csv")
X_test = pd.read_csv("../data/processed/X_test_v2.csv")
y_train = pd.read_csv("../data/processed/y_train_v2.csv")["target"]
y_test = pd.read_csv("../data/processed/y_test_v2.csv")["target"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {len(y_train)} ({y_train.mean()*100:.1f}% Hot)")
print(f"y_test:  {len(y_test)} ({y_test.mean()*100:.1f}% Hot)")

## 2. Modelo Baseline — Logistic Regression

La regresión logística funciona como línea base interpretable. Aunque no suele capturar relaciones no lineales complejas tan bien como los modelos basados en árboles, sí ofrece un punto de comparación útil para saber cuánto valor agrega la ingeniería de variables de V2.

**Qué observar:**
- si el baseline ya logra un ROC-AUC competitivo, entonces las features V2 son informativas por sí mismas
- si recall y precision quedan muy desbalanceados, puede indicar que los patrones no lineales siguen siendo importantes

**Diferencias V1 vs V2:**
En la V1 el baseline competía sobre una matriz one-hot mucho más fragmentada. En la V2 recibe features más densas y estadísticas, por lo que debería beneficiarse de una señal más estable.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print("=== LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")

## 3. Random Forest

Random Forest es una referencia robusta para problemas tabulares. Combina múltiples árboles y reduce varianza mediante bagging, por lo que suele comportarse bien incluso con relaciones no lineales y combinaciones entre variables.

**Qué observar:**
- si mejora el baseline sin disparar el sobreajuste
- si el ROC-AUC sube gracias a la interacción entre variables codificadas con Target Encoding
- qué tan bien balancea precisión y recall en comparación con Logistic Regression

**Diferencias V1 vs V2:**
En la V1 los árboles podían dispersar su capacidad en muchas columnas binarias. En la V2 trabajan sobre un espacio más compacto, por lo que sus particiones deberían ser más eficientes y estables.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

## 4. Gradient Boosting

Gradient Boosting construye árboles secuencialmente, corrigiendo los errores del árbol anterior. Esto lo vuelve especialmente útil cuando existen patrones complejos y efectos acumulativos que un modelo lineal no captura bien.

**Qué observar:**
- si supera a Random Forest en ROC-AUC
- si mantiene buen recall sobre la clase Hot sin deteriorar demasiado la precisión
- si las mejoras justifican el mayor costo computacional respecto al baseline

**Diferencias V1 vs V2:**
Con features densas y suavizadas, el boosting tiene más oportunidad de aprender patrones finos sin depender de reglas artificiales creadas por categorías raras.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               min_samples_leaf=10, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

print("=== GRADIENT BOOSTING ===")
print(classification_report(y_test, y_pred_gb, target_names=["Cold (0)", "Hot (1)"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_gb):.4f}")

## 5. LightGBM

LightGBM es un algoritmo de boosting optimizado para velocidad y escalabilidad. Se incluye como candidato avanzado porque suele rendir muy bien en datos tabulares cuando las features ya están bien preparadas.

**Qué observar:**
- si ofrece el mejor ROC-AUC del conjunto
- si mejora tiempos o calidad respecto a Gradient Boosting clásico
- si realmente aporta valor adicional frente al costo de mantener una dependencia extra

**Diferencias V1 vs V2:**
En la V1 su ventaja potencial se diluía por la fragmentación del espacio de features. En la V2 recibe una representación más compacta, por lo que su capacidad de ranking debería aprovecharse mejor.

**Nota:** Si `lightgbm` no está instalado, el notebook continúa sin fallar y simplemente omite este modelo.

In [ ]:
try:
    import lightgbm as lgb
    lgbm = lgb.LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.1,
                              min_child_samples=20, random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train)
    y_pred_lgbm = lgbm.predict(X_test)
    y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]
    print("=== LIGHTGBM ===")
    print(classification_report(y_test, y_pred_lgbm, target_names=["Cold (0)", "Hot (1)"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lgbm):.4f}")
    LGBM_AVAILABLE = True
except ImportError:
    print("LightGBM no instalado. Se omite este modelo.")
    print("Para instalar: pip install lightgbm")
    LGBM_AVAILABLE = False

## 6. Comparación de modelos — v2 vs v1

En esta sección consolidamos las métricas principales de todos los modelos. La comparación permite distinguir si la mejora de V2 proviene solo de tuning o si realmente la nueva representación de features está facilitando el aprendizaje.

**Qué observar:**
- qué modelo lidera en ROC-AUC, que es la métrica principal de ranking
- si el modelo ganador también mantiene un buen equilibrio entre precision, recall y F1
- si los modelos de árboles aprovechan mejor la representación densa de V2 que el baseline lineal

**Diferencias V1 vs V2:**
A diferencia de la V1, aquí los modelos de árboles trabajan con señales densas derivadas del Target Encoding en lugar de un espacio disperso de 0s y 1s. Esto debería traducirse en particiones más útiles, menor ruido y menor sensibilidad a categorías con poco soporte.

In [ ]:
results = {
    "Logistic Regression": {"y_pred": y_pred_lr, "y_proba": y_proba_lr, "model": lr},
    "Random Forest": {"y_pred": y_pred_rf, "y_proba": y_proba_rf, "model": rf},
    "Gradient Boosting": {"y_pred": y_pred_gb, "y_proba": y_proba_gb, "model": gb},
}
if LGBM_AVAILABLE:
    results["LightGBM"] = {"y_pred": y_pred_lgbm, "y_proba": y_proba_lgbm, "model": lgbm}

rows = []
for name, data in results.items():
    rows.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, data["y_pred"]),
        "Precision": precision_score(y_test, data["y_pred"]),
        "Recall": recall_score(y_test, data["y_pred"]),
        "F1": f1_score(y_test, data["y_pred"]),
        "ROC-AUC": roc_auc_score(y_test, data["y_proba"]),
    })

results_df = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
best_model_name = results_df.iloc[0]["Modelo"]
best_auc = results_df.iloc[0]["ROC-AUC"]

print("=== COMPARACIÓN DE MODELOS ===\n")
print(results_df.to_string(index=False, formatters={
    "Accuracy": "{:.4f}".format,
    "Precision": "{:.4f}".format,
    "Recall": "{:.4f}".format,
    "F1": "{:.4f}".format,
    "ROC-AUC": "{:.4f}".format,
}))
print(f"\nMejor modelo por ROC-AUC: {best_model_name} ({best_auc:.4f})")

display(results_df)

plot_df = results_df.melt(id_vars="Modelo", var_name="Métrica", value_name="Valor")
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=plot_df, x="Métrica", y="Valor", hue="Modelo")
plt.title("Comparación de métricas por modelo")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xlabel("")
plt.legend(title="Modelo", bbox_to_anchor=(1.02, 1), loc="upper left")
for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height):
        ax.annotate(f"{height:.3f}", (p.get_x() + p.get_width() / 2, height), ha="center", va="bottom", fontsize=8, rotation=90)
plt.tight_layout()
plt.show()

### Hallazgos observados

- **Gradient Boosting** obtiene el mejor **ROC-AUC = 0.7828**, con una ventaja marginal sobre **LightGBM = 0.7812**.
- **Random Forest** queda por debajo en ranking (**0.7561**), pero entrega el mejor **recall Hot = 0.6094**, lo que lo vuelve el modelo más agresivo para capturar oportunidades.
- **Logistic Regression** confirma que la V2 ya contiene señal útil incluso para un modelo lineal, pero sigue claramente por debajo de los modelos de árboles.
- La elección del modelo ganador no se apoya solo en accuracy: el criterio principal aquí es la **calidad de ranking** medida por ROC-AUC.

**Diferencias V1 vs V2:**

- En la **V1** la comparación estaba más condicionada por un espacio one-hot fragmentado; en la **V2** los árboles extraen valor de una representación densa y mucho más compacta.
- La brecha entre el baseline y los modelos de boosting sugiere que la **V2 conserva no linealidades relevantes sin inflar la dimensionalidad**.
- La lectura del tablero comparativo es más limpia: con solo **22 features finales**, el rendimiento refleja mejor la calidad del pipeline y menos el volumen artificial de columnas.


## 7. Curvas ROC

Las curvas ROC permiten comparar la capacidad de ranking de los modelos a distintos umbrales de decisión. Son especialmente útiles aquí porque el objetivo del proyecto no es solo clasificar, sino priorizar leads con mayor probabilidad de conversión.

**Hallazgos a validar:**
- un modelo dominante debería mantener su curva por encima de las demás en la mayor parte del rango
- diferencias pequeñas en accuracy pueden esconder mejoras relevantes en ranking, visibles en ROC-AUC
- si dos modelos son muy parecidos, conviene favorecer el más simple o más estable

**Diferencias V1 vs V2:**
En la V2 esperamos curvas más suaves y consistentes gracias a una representación menos ruidosa de las variables categóricas.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, data in results.items():
    RocCurveDisplay.from_predictions(y_test, data["y_proba"], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5)")
ax.set_title("Curvas ROC — Comparación de modelos (v2)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Hallazgos observados

- Las curvas ROC muestran que **Gradient Boosting** y **LightGBM** dominan casi todo el rango de umbrales y quedan claramente por encima de **Logistic Regression**.
- **Random Forest** mejora con claridad al baseline, pero no alcanza la capacidad de ranking de los modelos de boosting.
- La diferencia entre **Gradient Boosting** y **LightGBM** es muy pequeña; por eso, desde una perspectiva operativa, **Gradient Boosting** queda mejor posicionado por simplicidad de mantenimiento.

**Diferencias V1 vs V2:**

- Frente a la **V1**, aquí la lectura de las curvas es más estable porque el ranking ya no depende de cientos de dummies dispersas.
- La **V2** permite evaluar la calidad de ranking real de cada algoritmo y no su capacidad para memorizar categorías raras o demasiado específicas.
- La cercanía entre los dos modelos de boosting refuerza la idea de que la mejora ya proviene de la **representación de datos**, no solo del algoritmo.


## 8. Matrices de confusión

Las matrices de confusión complementan las métricas agregadas mostrando exactamente dónde se equivoca cada modelo. Aquí interesa especialmente entender el costo de perder Hot Leads reales frente a clasificar en exceso leads como Hot.

**Hallazgos a validar:**
- falsos negativos altos implican oportunidades comerciales perdidas
- falsos positivos altos implican priorización excesiva de leads fríos
- el mejor modelo no siempre será el de mayor accuracy, sino el que mejor balancee el objetivo comercial

**Diferencias V1 vs V2:**
Con señales más densas, la V2 debería producir fronteras de decisión menos frágiles y, por tanto, una matriz de confusión más equilibrada.

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
if n_models == 1:
    axes = [axes]
for ax, (name, data) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, data["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Cold", "Hot"], yticklabels=["Cold", "Hot"])
    ax.set_title(f"{name}")
    ax.set_ylabel("Real")
    ax.set_xlabel("Predicho")
plt.tight_layout()
plt.show()

### Hallazgos observados

- **Logistic Regression** produce una frontera conservadora: **TN=6600, FP=901, FN=2951, TP=1258**. Controla relativamente bien los falsos positivos, pero pierde demasiados Hot Leads reales.
- **Random Forest** es el modelo más agresivo: **TN=5570, FP=1931, FN=1644, TP=2565**. Recupera muchos más Hot Leads, aunque al costo de priorizar demasiados Cold como Hot.
- **Gradient Boosting** logra una matriz mucho más contenida en falsos positivos: **TN=7116, FP=385, FN=2457, TP=1752**. Es una política más precisa, aunque menos expansiva en recall.
- **LightGBM** empuja esa lógica un poco más: **TN=7232, FP=269, FN=2551, TP=1658**, quedando incluso más conservador que Gradient Boosting.
- La elección depende del objetivo de negocio: si se prioriza **capturar más Hot Leads**, Random Forest es más agresivo; si se prioriza **ranking limpio y menor ruido comercial**, el boosting queda mejor parado.

**Diferencias V1 vs V2:**

- En la **V1** las fronteras de decisión podían ser más frágiles por la dependencia de columnas binarias aisladas; en la **V2** el trade-off entre recall y precision se vuelve más explícito y defendible.
- La **V2** deja ver con más claridad qué modelos son agresivos y cuáles son conservadores, lo que facilita conectar la métrica con una decisión comercial real.


## 9. Importancia de features — v2 vs v1

Esta sección busca responder una pregunta clave: **qué señales está usando realmente el modelo para decidir**. En la V1, la importancia tendía a fragmentarse entre muchas columnas binarias derivadas del One-Hot Encoding. En la V2, al usar Target Encoding, las importancias deberían concentrarse en variables más compactas y estadísticamente más útiles.

**Resultados esperados:**
- menor fragmentación de importancia entre columnas redundantes
- señales más consistentes entre variables de negocio relevantes
- menor dependencia de categorías individuales con poco soporte

**Diferencias V1 vs V2:**
La V1 destacaba features binarias como `vehiculo_interes_KWID`; la V2 debería reflejar una lectura más rica del contexto del lead, con importancias distribuidas entre variables densas y temporales.

In [ ]:
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
elif hasattr(best_model, "coef_"):
    coef = np.abs(best_model.coef_).ravel()
    importances = pd.Series(coef, index=X_train.columns)
else:
    importances = None

if importances is not None:
    feat_imp = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    feat_imp.plot(kind="barh", color="#3498db", edgecolor="black", ax=ax)
    ax.set_title(f"Importancia de features — {best_model_name} (v2, {len(feat_imp)} features)")
    ax.set_xlabel("Importancia relativa")
    plt.tight_layout()
    plt.show()

    print(f"\nTop features v2 ({best_model_name}):")
    for feat, imp in feat_imp.tail(10).iloc[::-1].items():
        print(f"  {feat:45s} {imp:.4f}")

    print(f"\n{'='*65}")
    print("COMPARACIÓN CON v1")
    print(f"{'='*65}")
    print("v1: mayor fragmentación de importancia entre columnas one-hot")
    print("v2: importancias más compactas y comparables entre variables")
    print("\nTop 5 v2:")
    for feat, imp in feat_imp.tail(5).iloc[::-1].items():
        print(f"    {feat:42s} {imp:.4f}")
    print("\n→ La representación V2 reduce el sesgo por volumen y mejora la lectura del modelo")
else:
    print(f"El modelo {best_model_name} no expone importancias de features de forma nativa.")

### Hallazgos:

- La importancia en **Gradient Boosting** se concentra con fuerza en **mes_creacion (0.3685)**, seguida por **origen (0.1834)**, **dia_creacion (0.1790)** y **vehiculo_interes (0.1417)**.
- Después del bloque dominante aparecen variables de negocio plausibles como **concesionario**, **nombre_formulario**, **hora_creacion** y **campana**.
- La jerarquía resultante sugiere que el modelo está combinando **estacionalidad operativa**, **canal/origen** y **contexto del lead**, en lugar de depender de una sola categoría puntual.
- La cola de importancias es corta y legible, lo que mejora mucho la interpretabilidad del modelo ganador.

**Diferencias V1 vs V2:**

- En la **V1** la importancia tendía a fragmentarse entre dummies específicas; en la **V2** ya no aparece una columna aislada dominando por sí sola como ocurría con ciertas categorías one-hot.
- La **V2** resume mejor el contexto del lead con menos columnas y con nombres más cercanos a negocio, lo que facilita defender el modelo frente a usuarios no técnicos.
- Pasar a **22 features finales** reduce ruido, simplifica el análisis y hace más estable la lectura de importancias entre reentrenamientos.


## 10. Validación cruzada del mejor modelo

Después de identificar el mejor modelo en test, validamos su estabilidad con cross-validation sobre train. Esta verificación es importante para distinguir entre una buena métrica puntual y un comportamiento realmente consistente.

**Hallazgos a validar:**
- media alta de ROC-AUC confirma buen poder de discriminación
- desviación estándar baja sugiere que el modelo generaliza de forma estable
- si la varianza sube demasiado, conviene revisar sensibilidad a particiones o hiperparámetros

**Diferencias V1 vs V2:**
Con features más compactas y menos ruidosas, la V2 debería mostrar una varianza entre folds más contenida que un pipeline dependiente de muchas columnas binarias.

In [ ]:
print(f"=== VALIDACIÓN CRUZADA: {best_model_name} (5-fold) ===\n")
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
print(f"ROC-AUC por fold: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Media:  {cv_scores.mean():.4f}")
print(f"Std:    {cv_scores.std():.4f}")
if cv_scores.std() < 0.02:
    print("\nEl modelo es ESTABLE (baja varianza entre folds).")
else:
    print("\nADVERTENCIA: Varianza alta entre folds.")

### Hallazgos observados

- La validación cruzada del mejor modelo arroja **ROC-AUC medio = 0.7791** con **std = 0.0049**, una dispersión baja para un problema tabular real.
- La distancia entre la media de CV (**0.7791**) y el resultado en test (**0.7828**) es pequeña, lo que indica que el desempeño del hold-out no parece un accidente de partición.
- La conclusión operacional es favorable: el modelo ganador no solo rinde bien, sino que además lo hace de forma **estable**.

**Diferencias V1 vs V2:**

- La **V2** muestra una varianza entre folds muy contenida, consistente con un espacio de features más compacto y menos ruidoso.
- Esto fortalece la narrativa de que la mejora no viene solo del score puntual, sino de una **mejor generalización del pipeline completo**.


## 11. Guardar el mejor modelo

Finalmente persistimos el modelo ganador para reutilizarlo en inferencia dentro de la app Streamlit y en el pipeline productivo.

**Qué validar aquí:**
- que el archivo se guarde correctamente en `../models`
- que el tipo de modelo guardado coincida con el mejor resultado observado
- que el desempeño reportado en test quede documentado junto al artefacto

**Diferencias V1 vs V2:**
En la V2 el artefacto final queda asociado a un pipeline de features más compacto, consistente con la lógica de inferencia offline del proyecto.

In [ ]:
import joblib
import os

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = f"{MODEL_DIR}/best_model.joblib"
joblib.dump(best_model, model_path)

size_kb = os.path.getsize(model_path) / 1024
print(f"Modelo guardado: {model_path} ({size_kb:.1f} KB)")
print(f"Tipo: {type(best_model).__name__}")
print(f"ROC-AUC en test: {best_auc:.4f}")
print(f"Features: {X_train.shape[1]}")

### Hallazgos observados

- El artefacto final queda guardado como **`../models/best_model.joblib`**, con tamaño aproximado de **880.9 KB**.
- El tipo de modelo persistido coincide con el ganador del notebook: **GradientBoostingClassifier**.
- El artefacto queda asociado a **22 features** y a un **ROC-AUC en test de 0.7828**, lo que deja trazabilidad suficiente entre entrenamiento e inferencia.

**Diferencias V1 vs V2:**

- En la **V1** la narrativa final quedó más acoplada a artefactos y documentación legacy; en la **V2** el notebook cierra el ciclo con el mismo modelo que luego consume la app.
- La **V2** deja un artefacto más compacto y coherente con el pipeline de inferencia real, reduciendo el riesgo de desalineación entre entrenamiento, evaluación y despliegue.


---

### Conclusión del modelado (v2)

**Cambio principal:** One-Hot Encoding fragmentado en V1 → Target Encoding con Bayesian Smoothing en V2 para variables categóricas de alta cardinalidad.

**Resultado esperado:** mantener o mejorar la calidad predictiva mientras se reduce drásticamente la dimensionalidad, el sesgo por categorías pequeñas y el costo de entrenamiento.

**Hallazgos a consolidar tras la ejecución:**
- qué modelo logra el mejor ROC-AUC
- si la mejora viene acompañada de estabilidad en validación cruzada
- si la importancia de variables se vuelve más interpretable y menos dependiente de columnas binarias aisladas

**Lectura de negocio:**
Si la V2 mantiene métricas competitivas con menos features y mejor estabilidad, entonces no solo mejora el modelo: también mejora la confiabilidad del proceso completo de priorización de leads en producción.